<a href="https://colab.research.google.com/github/VinayaSharada/KateelLearningDemosToStudents/blob/main/TreasuryAnalytics/InvoiceLevelCollectionsPrediction/invoice_level_collections_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Invoice-Level Collections Prediction

This Colab notebook demonstrates a treasury collections workflow using synthetic invoice data, with two extensions built for real classroom use:

1. **Bring your own data.** Export a starter CSV template with the exact columns this notebook expects, fill it with your own ERP's invoice-level AR export, and re-run the same pipeline on real invoices instead of synthetic ones.
2. **Calendar-ready output.** Beyond a risk score, the notebook predicts *when* each invoice is expected to be paid and exports a day-by-day expected cash inflow feed you can load into a calendar or cash-forecasting tool.

Everything still runs with synthetic data by default — no setup required to see it work end to end.

## Learning goals

- Build invoice-level features from payment history, terms, disputes, relationship quality, and seasonality.
- Train a late-payment classifier **and** a days-late regressor, and explain why a calendar forecast needs both.
- Export a reusable data template so the same pipeline can run on a real ERP invoice export, not just synthetic data.
- Turn per-invoice predictions into a calendar-ready feed: which invoices are expected to be paid on which date, and the total expected inflow per date.
- Handle missing optional columns gracefully, and reject or flag bad required data explicitly, instead of assuming every real dataset looks like the synthetic one.
- Evaluate the payment-date prediction with RMSE on training vs. test data, and recognize what a train/test gap says about overfitting.
- Translate a measured accuracy improvement (RMSE reduction vs. a naive baseline) into an annual dollar value, and be explicit about the assumptions that number depends on.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime

rng = np.random.default_rng(42)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def dollar_axis(ax, axis="y"):
    formatter = mticker.FuncFormatter(lambda x, _pos: f"${x:,.0f}")
    (ax.yaxis if axis == "y" else ax.xaxis).set_major_formatter(formatter)

## 2. Bring your own data (optional)

Everything below runs on synthetic invoices by default. If you want to try this on your own accounts-receivable export instead:

1. Run the next cell to generate `invoice_data_template.csv` — a starter file with the exact columns this notebook expects, pre-filled with a few sample rows (including one row that deliberately leaves optional columns blank, to show that's allowed).
2. Download it (Colab downloads it automatically; in a local Jupyter environment, find it in the notebook's working folder), open it in Excel or Sheets, delete the sample rows, and paste in your own invoice-level export. Most ERPs (SAP, Oracle, NetSuite, Dynamics, QuickBooks, Zoho, Tally) can export invoice-level AR data with fields like these, sometimes under different column names you'll need to rename to match.
3. Set `USE_SYNTHETIC_DATA = False` in the toggle cell below and upload your filled file when prompted (Colab), or point `OWN_DATA_PATH` at it (local Jupyter).

Optional columns (`avg_days_beyond_terms`, `payment_history_score`, `open_dispute`, `relationship_strength`, `seasonality_stress`) don't need to be complete — if your ERP doesn't track them, leave them blank and the notebook fills in a neutral default, and tells you how many rows it had to default. That's intentional: it's the same "reduced confidence, not broken" pattern real AI tools should use when input data is incomplete, rather than silently pretending everything is known.

Required columns are handled more strictly, but still forgivingly: dates can be `YYYY-MM-DD`, `MM/DD/YYYY`, `DD-MM-YYYY`, or `DD/MM/YYYY`, and amounts can include currency symbols or thousands separators (`$420,000.00` parses the same as `420000`). A row with a missing ID, an unparseable date, or a missing/zero/negative amount can't be scored meaningfully — those rows are skipped and listed with a reason rather than silently dropped or crashing the whole notebook.

In [ ]:
TEMPLATE_COLUMNS = [
    "invoice_id", "customer_id", "customer_name", "industry", "region", "channel",
    "invoice_date", "due_date", "invoice_amount", "avg_days_beyond_terms",
    "payment_history_score", "open_dispute", "relationship_strength", "seasonality_stress",
]

TEMPLATE_SAMPLE_ROWS = [
    ["INV-1001", "CUST-0001", "Meridian Foods Pvt Ltd",   "Retail",        "North", "Distributor",  "2026-05-01", "2026-05-31", 420000,  12, 82, 0, "Stable",    "Low"],
    ["INV-1002", "CUST-0002", "Alcott Manufacturing",     "Manufacturing", "West",  "Field Sales",  "2026-05-04", "2026-06-18", 1250000, 28, 61, 1, "Weak",      "High"],
    ["INV-1003", "CUST-0003", "BrightPath Diagnostics",   "Healthcare",    "South", "Email",        "2026-05-10", "2026-06-09", 310000,  5,  91, 0, "Strategic", "Low"],
    ["INV-1004", "CUST-0004", "Nimbus Cloud Systems",     "Technology",    "East",  "Portal",       "2026-05-12", "2026-06-11", 875000,  "", "", "", "",        ""],
    ["INV-1005", "CUST-0001", "Meridian Foods Pvt Ltd",   "Retail",        "North", "Distributor",  "2026-05-20", "2026-06-19", 198000,  12, 82, 0, "Stable",    "Low"],
]

template_df = pd.DataFrame(TEMPLATE_SAMPLE_ROWS, columns=TEMPLATE_COLUMNS)
TEMPLATE_PATH = "invoice_data_template.csv"
template_df.to_csv(TEMPLATE_PATH, index=False)
print(f"Wrote {TEMPLATE_PATH} with {len(template_df)} sample rows (row INV-1004 shows optional columns left blank).")
template_df

In [ ]:
if IN_COLAB:
    colab_files.download(TEMPLATE_PATH)
else:
    print(f"Running outside Colab — find the template at: {TEMPLATE_PATH}")

### Column reference

| Column | Required? | Meaning | If you don't have it |
|---|---|---|---|
| `invoice_id` | Required | Unique invoice number | — |
| `customer_id` | Required | Stable customer key (survives name changes) | Use the customer name as a fallback key |
| `customer_name` | Required | Customer display name | — |
| `industry` | Required | Customer's industry segment | Use a single placeholder like `"General"` |
| `region` | Required | Sales region | Use a single placeholder |
| `channel` | Required | Sales / order channel | Use a single placeholder |
| `invoice_date` | Required | Invoice issue date (`YYYY-MM-DD`) | — |
| `due_date` | Required | Contractual due date (`YYYY-MM-DD`) | Compute as `invoice_date + payment terms` |
| `invoice_amount` | Required | Outstanding amount, in your own currency units | — |
| `avg_days_beyond_terms` | Optional | Customer's historical average days late | Leave blank — filled with the portfolio median |
| `payment_history_score` | Optional | Any internal 0-100 payment-reliability score | Leave blank — filled with a neutral 70 |
| `open_dispute` | Optional | 1 if this invoice is disputed, else 0 | Leave blank — assumed 0 (no dispute) |
| `relationship_strength` | Optional | `Weak` / `Stable` / `Strategic` | Leave blank — assumed `Stable` |
| `seasonality_stress` | Optional | `Low` / `Moderate` / `High` | Leave blank — assumed `Moderate` |

In [ ]:
USE_SYNTHETIC_DATA = True  # set to False after you've filled in and uploaded your own template
OWN_DATA_PATH = "invoice_data_template.csv"  # overwritten below if you upload a file in Colab

if not USE_SYNTHETIC_DATA:
    if IN_COLAB:
        print("Upload your filled invoice CSV:")
        uploaded = colab_files.upload()
        OWN_DATA_PATH = next(iter(uploaded)) if uploaded else OWN_DATA_PATH
    print(f"Will read invoices to score from: {OWN_DATA_PATH}")
else:
    print("Using synthetic data. Set USE_SYNTHETIC_DATA = False above to use your own file instead.")

## 3. Generate labeled training data

The model needs historical examples where we already know what actually happened (was the invoice late? by how many days?) to learn from. Synthetic data stands in for that here so the notebook works with zero setup — in a real deployment, you'd train on your own AR aging history instead, going back a year or more.

In [ ]:
INDUSTRIES = ["Technology", "Retail", "Manufacturing", "Healthcare"]
REGIONS = ["North", "South", "East", "West"]
CHANNELS = ["Email", "Portal", "Distributor", "Field Sales"]
TERMS_CHOICES = [15, 30, 45, 60]

INDUSTRY_STRESS = {"Technology": 0.10, "Retail": 0.18, "Manufacturing": 0.14, "Healthcare": 0.08}
SEASONALITY_WEIGHT = {"Low": 0.0, "Moderate": 0.08, "High": 0.16}
RELATIONSHIP_WEIGHT = {"Weak": 0.18, "Stable": 0.06, "Strategic": -0.03}
CHANNEL_WEIGHT = {"Email": 0.04, "Portal": 0.01, "Distributor": 0.09, "Field Sales": -0.02}


def generate_customers(n_customers):
    avg_days_beyond_terms = np.clip(rng.normal(12, 9, size=n_customers), 0, 60)
    payment_history_score = np.clip(100 - avg_days_beyond_terms * 1.7 + rng.normal(0, 8, size=n_customers), 25, 98)
    first = rng.choice(["Meridian", "Alcott", "BrightPath", "Nimbus", "Coastal", "Summit", "Kessler", "Vantage", "Harbor", "Falcon"], size=n_customers)
    second = rng.choice(["Foods", "Manufacturing", "Diagnostics", "Cloud Systems", "Traders", "Logistics", "Retail Group", "Industries", "Textiles", "Distribution"], size=n_customers)
    return pd.DataFrame({
        "customer_id": [f"CUST-{i:04d}" for i in range(1, n_customers + 1)],
        "customer_name": [f"{a} {b}" for a, b in zip(first, second)],
        "industry": rng.choice(INDUSTRIES, size=n_customers, p=[0.3, 0.25, 0.25, 0.2]),
        "region": rng.choice(REGIONS, size=n_customers),
        "relationship_strength": rng.choice(["Weak", "Stable", "Strategic"], size=n_customers, p=[0.2, 0.55, 0.25]),
        "avg_days_beyond_terms": np.round(avg_days_beyond_terms, 1),
        "payment_history_score": np.round(payment_history_score, 1),
    })


def generate_invoices(customers, n_invoices, as_of, labeled):
    picks = customers.iloc[rng.integers(0, len(customers), size=n_invoices)].reset_index(drop=True)
    channel = rng.choice(CHANNELS, size=n_invoices, p=[0.25, 0.3, 0.2, 0.25])
    payment_terms = rng.choice(TERMS_CHOICES, size=n_invoices, p=[0.15, 0.45, 0.25, 0.15])
    invoice_amount = np.round(rng.lognormal(mean=13.2, sigma=0.7, size=n_invoices), 2)
    open_dispute = rng.binomial(1, 0.14, size=n_invoices)
    seasonality_stress = rng.choice(["Low", "Moderate", "High"], size=n_invoices, p=[0.45, 0.35, 0.2])
    invoice_offset_days = rng.integers(5, 45, size=n_invoices)
    invoice_date = pd.Timestamp(as_of) - pd.to_timedelta(invoice_offset_days, unit="D")
    due_date = invoice_date + pd.to_timedelta(payment_terms, unit="D")

    df = picks.drop(columns=[]).copy()
    df["invoice_id"] = [f"INV-{20000 + i}" for i in range(n_invoices)]
    df["channel"] = channel
    df["invoice_date"] = invoice_date
    df["due_date"] = due_date
    df["invoice_amount"] = invoice_amount
    df["open_dispute"] = open_dispute
    df["seasonality_stress"] = seasonality_stress

    if labeled:
        industry_stress = df["industry"].map(INDUSTRY_STRESS).to_numpy()
        seasonality_weight = df["seasonality_stress"].map(SEASONALITY_WEIGHT).to_numpy()
        relationship_weight = df["relationship_strength"].map(RELATIONSHIP_WEIGHT).to_numpy()
        channel_weight = df["channel"].map(CHANNEL_WEIGHT).to_numpy()

        risk_signal = (
            0.025 * df["avg_days_beyond_terms"].to_numpy()
            - 0.018 * (df["payment_history_score"].to_numpy() / 10)
            + 0.22 * df["open_dispute"].to_numpy()
            + industry_stress + seasonality_weight + relationship_weight + channel_weight
            + rng.normal(0, 0.08, size=n_invoices)
        )
        df["_late_payment"] = (risk_signal > np.quantile(risk_signal, 0.56)).astype(int)
        # Continuous ground truth: how many days late (negative = paid early), used to
        # train a regressor for the calendar forecast — a classifier alone can't tell you *when*.
        actual_days_late = df["avg_days_beyond_terms"].to_numpy() * 0.6 + risk_signal * 18 + rng.normal(0, 4, size=n_invoices)
        df["_actual_days_late"] = np.round(np.clip(actual_days_late, -10, 90)).astype(int)

    return df.reset_index(drop=True)


AS_OF = pd.Timestamp.today().normalize()
customers = generate_customers(n_customers=220)
train_raw = generate_invoices(customers, n_invoices=1200, as_of=AS_OF, labeled=True)
print(f"Generated {len(train_raw)} labeled training invoices across {customers.shape[0]} synthetic customers, as of {AS_OF.date()}.")
train_raw.head()

## 4. Engineer features and train two models

A risk score alone can't tell you *when* an invoice is expected to be paid — only *whether* it's likely to be late. To build a calendar feed we need both: a classifier for late-payment risk (as before) and a regressor that predicts how many days late (or early) an invoice will land relative to its due date.

`prepare_features()` is the one function used on both the synthetic training data and whatever you upload, so a real ERP export goes through the exact same handling as the teaching data — no special case. It does three things:

1. **Parses dates and amounts defensively.** Real exports use different date formats (`2026-05-01`, `05/01/2026`, `01-05-2026`) and often format amounts with currency symbols or thousands separators (`$420,000.00`). Both are handled per-value, so one inconsistent cell doesn't break the whole column.
2. **Drops and reports unusable rows instead of crashing.** A row with a missing `invoice_id`, an unparseable date, or a missing/zero/negative `invoice_amount` can't be scored meaningfully — it's pulled into an `errors` table with a reason, and the rest of the file still runs.
3. **Defaults missing optional columns**, exactly as described in the column reference table above.

In [ ]:
NUMERIC_FEATURES = ["payment_terms", "invoice_amount", "avg_days_beyond_terms", "payment_history_score", "open_dispute"]
CATEGORICAL_FEATURES = ["industry", "region", "channel", "relationship_strength", "seasonality_stress"]
REQUIRED_COLUMNS = ["invoice_id", "customer_id", "customer_name", "industry", "region", "channel", "invoice_date", "due_date", "invoice_amount"]
DATE_FORMATS = ("%Y-%m-%d", "%m/%d/%Y", "%d-%m-%Y", "%d/%m/%Y", "%Y/%m/%d")


def parse_date_flexible(value):
    """Tries a handful of common date formats so real ERP exports parse the
    same as the synthetic ISO-format dates. Returns NaT (not an exception)
    for anything unparseable, so one bad cell doesn't take down the column.
    Handles both real datetime/Timestamp values (e.g. from the synthetic
    generator) and plain strings (e.g. from a CSV upload)."""
    if pd.isna(value):
        return pd.NaT
    if isinstance(value, (pd.Timestamp, datetime)):
        return pd.Timestamp(value)
    text = str(value).strip()
    for fmt in DATE_FORMATS:
        try:
            return pd.Timestamp(datetime.strptime(text, fmt))
        except ValueError:
            continue
    return pd.NaT


def parse_amount_flexible(value):
    """Strips currency symbols and thousands separators, so "$420,000.00" and
    420000 both parse to the same number."""
    if pd.isna(value):
        return np.nan
    text = str(value).strip()
    for symbol in ("$", "₹", ",", "USD", "INR"):
        text = text.replace(symbol, "")
    text = text.strip()
    try:
        return float(text) if text else np.nan
    except ValueError:
        return np.nan


def prepare_features(raw_df):
    """Returns (clean_df, defaults_used, quality_warnings, errors_df).
    clean_df is ready to feed to the model; errors_df lists rows that were
    dropped and why, so nothing fails silently."""
    missing_required = [c for c in REQUIRED_COLUMNS if c not in raw_df.columns]
    if missing_required:
        raise ValueError(f"Missing required column(s): {missing_required}. See the column reference table above.")

    df = raw_df.copy().reset_index(drop=True)
    df["invoice_date"] = df["invoice_date"].apply(parse_date_flexible)
    df["due_date"] = df["due_date"].apply(parse_date_flexible)
    df["invoice_amount"] = df["invoice_amount"].apply(parse_amount_flexible)

    reason = pd.Series([None] * len(df), index=df.index, dtype=object)
    blank_id = df["invoice_id"].isna() | (df["invoice_id"].astype(str).str.strip() == "")
    reason[blank_id] = "missing invoice_id"
    reason[reason.isna() & df["invoice_date"].isna()] = "unparseable invoice_date"
    reason[reason.isna() & df["due_date"].isna()] = "unparseable due_date"
    reason[reason.isna() & df["invoice_amount"].isna()] = "missing or unparseable invoice_amount"
    reason[reason.isna() & (df["invoice_amount"] <= 0)] = "invoice_amount is zero or negative"

    errors_df = df.loc[reason.notna()].assign(_error_reason=reason[reason.notna()])
    clean_df = df.loc[reason.isna()].copy().reset_index(drop=True)

    if clean_df.empty:
        raise ValueError("No valid invoices remain after data-quality checks — see the error rows above before re-running.")

    clean_df["payment_terms"] = (clean_df["due_date"] - clean_df["invoice_date"]).dt.days

    quality_warnings = []
    swapped = clean_df["payment_terms"] < 0
    if swapped.any():
        quality_warnings.append(f"{int(swapped.sum())} row(s) have due_date before invoice_date (negative payment terms) — check for swapped columns.")
    duplicate_ids = int(clean_df["invoice_id"].duplicated().sum())
    if duplicate_ids:
        quality_warnings.append(f"{duplicate_ids} duplicate invoice_id value(s) found — each will still be scored, but confirm that's intentional.")

    defaults_used = {}
    clean_df["avg_days_beyond_terms"] = pd.to_numeric(clean_df.get("avg_days_beyond_terms"), errors="coerce")
    fallback_dbt = clean_df["avg_days_beyond_terms"].median()
    fallback_dbt = 15.0 if pd.isna(fallback_dbt) else fallback_dbt
    defaults_used["avg_days_beyond_terms"] = int(clean_df["avg_days_beyond_terms"].isna().sum())
    clean_df["avg_days_beyond_terms"] = clean_df["avg_days_beyond_terms"].fillna(fallback_dbt)

    clean_df["payment_history_score"] = pd.to_numeric(clean_df.get("payment_history_score"), errors="coerce")
    defaults_used["payment_history_score"] = int(clean_df["payment_history_score"].isna().sum())
    clean_df["payment_history_score"] = clean_df["payment_history_score"].fillna(70)

    clean_df["open_dispute"] = pd.to_numeric(clean_df.get("open_dispute"), errors="coerce")
    defaults_used["open_dispute"] = int(clean_df["open_dispute"].isna().sum())
    clean_df["open_dispute"] = clean_df["open_dispute"].fillna(0).astype(int)

    if "relationship_strength" not in clean_df.columns:
        clean_df["relationship_strength"] = np.nan
    defaults_used["relationship_strength"] = int(clean_df["relationship_strength"].isna().sum())
    clean_df["relationship_strength"] = clean_df["relationship_strength"].fillna("Stable")

    if "seasonality_stress" not in clean_df.columns:
        clean_df["seasonality_stress"] = np.nan
    defaults_used["seasonality_stress"] = int(clean_df["seasonality_stress"].isna().sum())
    clean_df["seasonality_stress"] = clean_df["seasonality_stress"].fillna("Moderate")

    return clean_df, defaults_used, quality_warnings, errors_df


preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), NUMERIC_FEATURES),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CATEGORICAL_FEATURES),
])

clf_pipeline = Pipeline([("prep", preprocessor), ("clf", RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42))])
reg_pipeline = Pipeline([("prep", preprocessor), ("reg", RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42))])

train_df, _, _, train_errors = prepare_features(train_raw)
if not train_errors.empty:
    print(f"Note: {len(train_errors)} synthetic training row(s) failed data-quality checks — this should be 0 for synthetic data; investigate if it isn't.")

X = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y_class = train_df["_late_payment"]
y_days = train_df["_actual_days_late"]

X_train, X_test, yc_train, yc_test, yd_train, yd_test = train_test_split(
    X, y_class, y_days, test_size=0.25, random_state=42, stratify=y_class
)

clf_pipeline.fit(X_train, yc_train)
reg_pipeline.fit(X_train, yd_train)

proba_test = clf_pipeline.predict_proba(X_test)[:, 1]
days_pred_test = reg_pipeline.predict(X_test)

print("Late-payment classifier — ROC AUC:", round(roc_auc_score(yc_test, proba_test), 3))
print(classification_report(yc_test, clf_pipeline.predict(X_test)))
print("Days-late regressor — Mean Absolute Error:", round(mean_absolute_error(yd_test, days_pred_test), 1), "days")

### How accurate is the payment-date prediction? RMSE, train vs test

Two things worth checking before trusting the calendar feed:

1. **RMSE vs MAE.** MAE (printed above) is the average absolute error in days. RMSE squares errors before averaging, so it penalizes large misses harder — a model that's mostly right but occasionally very wrong will look fine on MAE and much worse on RMSE.
2. **Train vs test.** If train error is much lower than test error, the model has memorized the training data rather than learned something that generalizes — a classic overfitting signal. A model you'd trust on new invoices should have train and test error reasonably close.

The table below also includes a **naive baseline**: always assuming the invoice pays exactly on its due date (0 days late). That's the same assumption the "without AI" calendar in Section 7 makes. The gap between the naive row and the AI regressor row is the actual accuracy the model is contributing — and it's what the economic-value calculation in Section 8 is built on.

In [ ]:
def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred) ** 0.5


days_pred_train = reg_pipeline.predict(X_train)

# Naive baseline: "trust the due date" is the same as always predicting 0 days late/early.
naive_pred_train = np.zeros_like(yd_train)
naive_pred_test = np.zeros_like(yd_test)

accuracy_table = pd.DataFrame([
    {"model": "Naive (assume due date)", "split": "train", "rmse_days": rmse(yd_train, naive_pred_train), "mae_days": mean_absolute_error(yd_train, naive_pred_train)},
    {"model": "Naive (assume due date)", "split": "test", "rmse_days": rmse(yd_test, naive_pred_test), "mae_days": mean_absolute_error(yd_test, naive_pred_test)},
    {"model": "AI regressor", "split": "train", "rmse_days": rmse(yd_train, days_pred_train), "mae_days": mean_absolute_error(yd_train, days_pred_train)},
    {"model": "AI regressor", "split": "test", "rmse_days": rmse(yd_test, days_pred_test), "mae_days": mean_absolute_error(yd_test, days_pred_test)},
]).round(2)

train_test_gap = accuracy_table.loc[(accuracy_table["model"] == "AI regressor") & (accuracy_table["split"] == "test"), "rmse_days"].iloc[0] - accuracy_table.loc[(accuracy_table["model"] == "AI regressor") & (accuracy_table["split"] == "train"), "rmse_days"].iloc[0]
if train_test_gap > 1.5:
    print(f"Note: test RMSE is {train_test_gap:.1f} days worse than train RMSE — check for overfitting before trusting this on new invoices.")

accuracy_table

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
for ax, (y_true, y_pred, title) in zip(axes, [(yd_train, days_pred_train, "Training set"), (yd_test, days_pred_test, "Test set")]):
    ax.scatter(y_true, y_pred, alpha=0.3, s=12, color="#2563eb")
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, color="#94a3b8", linestyle="--", label="Perfect prediction")
    ax.set_xlabel("Actual days late")
    ax.set_ylabel("Predicted days late")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Score the invoices you want to act on

This is the step that switches between synthetic data and your own upload. The model was trained above (it needs labeled history); this step only *applies* it — real invoices you upload here never need a "was it actually late" column, because that's exactly what we're trying to predict.

Whichever source you use, the same `prepare_features()` validation runs against it. The data-quality report printed below is what makes the output trustworthy: it tells you exactly how many rows were scored, how many were dropped and why, and how many optional fields were defaulted — instead of silently producing a calendar feed you can't fully trust.

In [ ]:
if USE_SYNTHETIC_DATA:
    apply_raw = generate_invoices(customers, n_invoices=150, as_of=AS_OF, labeled=False)
    print(f"Generated {len(apply_raw)} synthetic invoices (this week's open AR, for demo purposes).")
else:
    apply_raw = pd.read_csv(OWN_DATA_PATH)
    print(f"Loaded {len(apply_raw)} invoices from {OWN_DATA_PATH}.")

if len(apply_raw) == 0:
    raise ValueError(f"{OWN_DATA_PATH} has no data rows — check the file before continuing.")

apply_df, defaults_used, quality_warnings, apply_errors = prepare_features(apply_raw)

print(f"\nData quality report: {len(apply_df)} of {len(apply_raw)} rows will be scored.")

if not apply_errors.empty:
    print(f"  {len(apply_errors)} row(s) skipped (not scored):")
    for _, row in apply_errors.head(10).iterrows():
        row_id = row.get("invoice_id")
        row_label = "(blank invoice_id)" if pd.isna(row_id) or str(row_id).strip() == "" else row_id
        print(f"    - {row_label}: {row['_error_reason']}")
    if len(apply_errors) > 10:
        print(f"    ... and {len(apply_errors) - 10} more — see the apply_errors dataframe.")

rows_defaulted = {k: v for k, v in defaults_used.items() if v > 0}
if rows_defaulted:
    print("  Optional columns defaulted (see column reference table):")
    for col, n_rows in rows_defaulted.items():
        print(f"    - {col}: {n_rows} of {len(apply_df)} rows")
else:
    print("  All optional columns were present — no defaults were needed.")

for warning in quality_warnings:
    print(f"  Warning: {warning}")

apply_features = apply_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
apply_df["late_risk_probability"] = clf_pipeline.predict_proba(apply_features)[:, 1].round(3)
apply_df["predicted_days_vs_due"] = reg_pipeline.predict(apply_features).round().astype(int)
apply_df["cash_at_risk"] = apply_df["invoice_amount"] * apply_df["late_risk_probability"]
apply_df["priority_band"] = pd.cut(apply_df["late_risk_probability"], bins=[0, 0.35, 0.6, 1], labels=["Monitor", "Targeted follow-up", "Immediate action"])

apply_df[["invoice_id", "customer_name", "invoice_amount", "due_date", "late_risk_probability", "predicted_days_vs_due", "priority_band"]].head(10)

## 6. Collections priority view

Same idea as a standard collections worklist: rank by risk-weighted dollar exposure so the highest-value, highest-risk invoices surface first.

In [ ]:
priority_view = apply_df.sort_values(["late_risk_probability", "invoice_amount"], ascending=[False, False]).head(15)
priority_view[["invoice_id", "customer_name", "industry", "region", "channel", "invoice_amount", "due_date", "late_risk_probability", "predicted_days_vs_due", "priority_band"]]

## 7. Calendar-ready expected inflow feed: without AI vs. with AI

This is the payoff: not just "this invoice is risky," but **"this invoice is expected to be paid on this date"** — and rolled up, **"here is the total cash expected to land on each calendar date."**

To make the AI's contribution concrete rather than assumed, the notebook builds the calendar **two ways** from the same invoices:

- **Without AI** — the naive forecast a treasury team gets for free from any ERP: assume every invoice pays exactly on its contractual `due_date`.
- **With AI** — `expected_payment_date` shifts each invoice by the regressor's predicted days late (or early, if negative), using the risk signals from Section 4.

Comparing the two is the actual point: if they rarely diverge, the model isn't earning its complexity. If they diverge a lot, that gap is what the AI is contributing to the forecast — and worth checking against the regressor's Mean Absolute Error from Section 4 before trusting it.

In [ ]:
apply_df["expected_payment_date"] = apply_df["due_date"] + pd.to_timedelta(apply_df["predicted_days_vs_due"], unit="D")

calendar_feed = apply_df[[
    "invoice_id", "customer_name", "invoice_amount", "due_date",
    "predicted_days_vs_due", "expected_payment_date", "late_risk_probability", "priority_band",
]].sort_values("expected_payment_date").reset_index(drop=True)


def daily_totals(df, date_col):
    return (
        df.assign(date=df[date_col].dt.date)
        .groupby("date")
        .agg(expected_inflow=("invoice_amount", "sum"), invoice_count=("invoice_id", "count"))
        .reset_index()
        .sort_values("date")
    )


daily_inflow_without_ai = daily_totals(calendar_feed, "due_date")             # naive: contractual due date only
daily_inflow_with_ai = daily_totals(calendar_feed, "expected_payment_date")   # AI-adjusted: regressor-shifted date

print(f"Without AI (due date):        {calendar_feed['due_date'].min().date()} to {calendar_feed['due_date'].max().date()}, {len(daily_inflow_without_ai)} distinct dates.")
print(f"With AI (predicted payment):  {calendar_feed['expected_payment_date'].min().date()} to {calendar_feed['expected_payment_date'].max().date()}, {len(daily_inflow_with_ai)} distinct dates.")
calendar_feed.head(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharey=True)

axes[0].bar(daily_inflow_without_ai["date"].astype(str), daily_inflow_without_ai["expected_inflow"], color="#94a3b8")
dollar_axis(axes[0])
axes[0].set_title("Without AI — assumes every invoice pays exactly on its due date")
axes[0].tick_params(axis="x", labelrotation=60, labelsize=7)

axes[1].bar(daily_inflow_with_ai["date"].astype(str), daily_inflow_with_ai["expected_inflow"], color="#2563eb")
dollar_axis(axes[1])
axes[1].set_title("With AI — shifted by the regressor's predicted days late/early")
axes[1].tick_params(axis="x", labelrotation=60, labelsize=7)

fig.suptitle("Total Expected Inflow by Calendar Date", y=1.0)
plt.tight_layout()
plt.show()

### How much does the AI prediction actually change?

A visual comparison is a start, but a number is easier to argue with. This compares total expected inflow in the near term under each method — the gap between them is what a treasury team would be trusting the model for.

In [ ]:
HORIZON_DAYS = 14
horizon_start = pd.Timestamp(AS_OF).normalize()
horizon_end = horizon_start + pd.Timedelta(days=HORIZON_DAYS)

without_ai_near_term = calendar_feed.loc[calendar_feed["due_date"].between(horizon_start, horizon_end), "invoice_amount"].sum()
with_ai_near_term = calendar_feed.loc[calendar_feed["expected_payment_date"].between(horizon_start, horizon_end), "invoice_amount"].sum()
difference = with_ai_near_term - without_ai_near_term

comparison_summary = pd.DataFrame({
    "metric": [
        f"Expected inflow in next {HORIZON_DAYS} days — without AI (due date)",
        f"Expected inflow in next {HORIZON_DAYS} days — with AI (predicted date)",
        "Difference (with AI minus without AI)",
    ],
    "value": [f"${without_ai_near_term:,.0f}", f"${with_ai_near_term:,.0f}", f"${difference:,.0f}"],
})
comparison_summary

### Export the calendar feed

`calendar_feed` already has both dates side by side (`due_date` and `expected_payment_date`), so a single invoice-level export supports both views. Separate daily-aggregate CSVs are provided for each, plus an optional Google Calendar-style import file (`Subject`, `Start Date`, `Description` columns) built from the AI-adjusted dates. Calendar import formats change across providers — check your target tool's current import documentation before relying on the last one.

In [ ]:
CALENDAR_FEED_PATH = "expected_payment_calendar.csv"
DAILY_INFLOW_WITHOUT_AI_PATH = "daily_expected_inflow_without_ai.csv"
DAILY_INFLOW_WITH_AI_PATH = "daily_expected_inflow_with_ai.csv"
GCAL_IMPORT_PATH = "collections_calendar_import.csv"

calendar_feed.to_csv(CALENDAR_FEED_PATH, index=False)
daily_inflow_without_ai.to_csv(DAILY_INFLOW_WITHOUT_AI_PATH, index=False)
daily_inflow_with_ai.to_csv(DAILY_INFLOW_WITH_AI_PATH, index=False)

gcal_import = pd.DataFrame({
    "Subject": calendar_feed.apply(lambda r: f"Expected payment: {r['customer_name']} ({r['invoice_id']}) ${r['invoice_amount']:,.0f}", axis=1),
    "Start Date": calendar_feed["expected_payment_date"].dt.strftime("%m/%d/%Y"),
    "All Day Event": "True",
    "Description": calendar_feed.apply(lambda r: f"Invoice {r['invoice_id']}, {r['customer_name']}, late-risk {r['late_risk_probability']:.0%}, priority: {r['priority_band']}", axis=1),
})
gcal_import.to_csv(GCAL_IMPORT_PATH, index=False)

print(f"Wrote {CALENDAR_FEED_PATH}, {DAILY_INFLOW_WITHOUT_AI_PATH}, {DAILY_INFLOW_WITH_AI_PATH}, and {GCAL_IMPORT_PATH}.")

if IN_COLAB:
    colab_files.download(CALENDAR_FEED_PATH)
    colab_files.download(DAILY_INFLOW_WITHOUT_AI_PATH)
    colab_files.download(DAILY_INFLOW_WITH_AI_PATH)
    colab_files.download(GCAL_IMPORT_PATH)
else:
    print("Running outside Colab — find the four files above in the notebook's working folder.")

## 8. Economic value of the AI's forecast accuracy

A more accurate payment-date forecast only matters if it changes a real decision. The standard treasury framing: **forecast uncertainty forces you to hold a cash buffer** — idle cash, or unused credit-line headroom — sized to cover how wrong your forecast could be. A tighter forecast (lower RMSE) needs a smaller buffer, freeing capital that would otherwise sit idle or draw a commitment fee.

This is a simplification — real treasury policy might size buffers using a volatility multiple, Value-at-Risk, or stress scenarios rather than RMSE directly — but it turns "the AI adds value" from a vague claim into a specific number you can argue with.

**The math:**
- `buffer_reduction_days` = naive forecast's test-set RMSE − AI forecast's test-set RMSE (from the table in Section 4)
- `avg_daily_inflow` = today's scored portfolio value ÷ the number of days it spans
- `capital_freed` = `buffer_reduction_days` × `avg_daily_inflow`
- `annual_value` = `capital_freed` × an annual cost-of-capital rate (your treasury's short-term borrowing rate, or cost of funds)

In [ ]:
COST_OF_CAPITAL_RATE = 0.09  # annual short-term borrowing / cost-of-capital rate -- replace with your own

naive_test_rmse = accuracy_table.loc[(accuracy_table["model"] == "Naive (assume due date)") & (accuracy_table["split"] == "test"), "rmse_days"].iloc[0]
ai_test_rmse = accuracy_table.loc[(accuracy_table["model"] == "AI regressor") & (accuracy_table["split"] == "test"), "rmse_days"].iloc[0]
buffer_reduction_days = naive_test_rmse - ai_test_rmse

if buffer_reduction_days < 0:
    print("Warning: the naive due-date assumption was MORE accurate than the AI regressor on this test set.")
    print("That is a real result, not a bug -- it means this model should not be trusted for forecasting yet.")

forecast_horizon_days = max((calendar_feed["due_date"].max() - calendar_feed["due_date"].min()).days, 1)
avg_daily_inflow = calendar_feed["invoice_amount"].sum() / forecast_horizon_days

capital_freed = buffer_reduction_days * avg_daily_inflow
annual_value = capital_freed * COST_OF_CAPITAL_RATE

economic_value_summary = pd.DataFrame({
    "metric": [
        "Naive forecast RMSE (test set)",
        "AI forecast RMSE (test set)",
        "Forecast accuracy improvement",
        "Average daily inflow (today's scored portfolio)",
        "Capital freed by a tighter buffer",
        f"Annual economic value (at {COST_OF_CAPITAL_RATE:.0%} cost of capital)",
    ],
    "value": [
        f"{naive_test_rmse:.1f} days",
        f"{ai_test_rmse:.1f} days",
        f"{buffer_reduction_days:.1f} fewer days of uncertainty",
        f"${avg_daily_inflow:,.0f}",
        f"${capital_freed:,.0f}",
        f"${annual_value:,.0f}",
    ],
})
economic_value_summary

**Read this as a discussion starting point, not a boardroom-ready number.** It depends on: (1) RMSE being a reasonable proxy for the buffer a treasury team would actually hold, (2) the cost-of-capital rate set above, and (3) today's scored portfolio being representative of ongoing volume. Change `COST_OF_CAPITAL_RATE` to your own organization's actual short-term borrowing rate and re-run before using this number in a real conversation.

## 9. CFO summary

The same translation from the chat-based demo: what does a day of DSO improvement, or the highest-risk slice of AR, actually mean in cash terms.

In [ ]:
total_invoiced = apply_df["invoice_amount"].sum()
cash_per_dso_day = total_invoiced / 365
high_risk_share = apply_df.loc[apply_df["priority_band"] == "Immediate action", "invoice_amount"].sum() / total_invoiced

summary = pd.DataFrame({
    "metric": [
        "Total invoice value scored",
        "Cash freed if DSO improves by 1 day",
        "Share of value in immediate-action band",
    ],
    "value": [
        f"${total_invoiced:,.0f}",
        f"${cash_per_dso_day:,.0f}",
        f"{high_risk_share * 100:.1f}%",
    ],
})
summary

## Discussion prompts

- Which features look operationally actionable versus merely descriptive?
- When should a strategic relationship lower collections urgency even if a model predicts delay?
- How would you connect the priority table to DSO improvement and working-capital release targets?
- Compare the two calendar charts (without AI vs. with AI). Where do they diverge most, and what in the invoice data (dispute flag, customer history, relationship strength) would explain that divergence in business terms?
- Look at the near-term comparison number — is the AI forecast more or less optimistic than the naive due-date forecast? Which one would you actually want to plan short-term borrowing around, and why?
- Are there dates where expected inflow is unusually concentrated in either version? What would that mean for short-term cash planning if the biggest customer on that date actually slips?
- If you exported `expected_payment_calendar.csv` into your own treasury team's forecasting tool, what would you still need to validate before trusting the total daily inflow number in a real decision?
- The regressor's Mean Absolute Error (printed above) is measured in days. How would you explain that error margin to someone deciding which of the two calendar views to trust?
- Look at the RMSE table in Section 4 — is train RMSE close to test RMSE, or is there a gap that suggests overfitting? Would you deploy this model as-is?
- The economic-value calculation in Section 8 depends entirely on `COST_OF_CAPITAL_RATE` and the RMSE-to-buffer assumption. Change the rate to something you think is more realistic for your context and see how much the annual value estimate moves — how confident are you in the number now?
- If you tried this with your own data: how many rows did the data-quality report skip, and were the reasons about genuinely bad data, or about this notebook's format assumptions not matching your ERP's export? What would you fix on which side?